# B5 — Customer Segmentation: K-Means and elbow method

**Laboratory Practice III (410246), Group B**

Dataset reference: https://www.kaggle.com/search?q=sales_data_sample.csv

Run cells top-to-bottom. Set `CSV_PATH` to your downloaded CSV file to use real data; `None` runs the included synthetic demonstration. Results on synthetic data are not real-world model performance.

In [ ]:
# Set to a local CSV filename/path after downloading the Kaggle dataset.
# Keep None to run with built-in synthetic demonstration data.
CSV_PATH = None  # Example: 'sales_data_sample.csv'

## Implementation and evaluation

In [ ]:
"""B5: Sales customer segmentation with K-Means and elbow method."""

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

import matplotlib
# Notebook runs in an interactive environment.
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def main():
    if CSV_PATH:  # Load sales_data_sample.csv here.
        df=pd.read_csv(CSV_PATH,encoding='latin-1')
    else:
        rng=np.random.default_rng(42);n=240
        df=pd.DataFrame({'CUSTOMERNAME':np.repeat([f'Customer_{i}' for i in range(60)],4),
                         'SALES':rng.uniform(100,10000,n),'QUANTITYORDERED':rng.integers(1,50,n),'ORDERNUMBER':np.arange(n)})
        print('Using synthetic demonstration data.')
    if {'CUSTOMERNAME','SALES'}<=set(df):
        df['SALES']=pd.to_numeric(df['SALES'],errors='coerce')
        data=df.groupby('CUSTOMERNAME').agg(total_sales=('SALES','sum'),orders=('SALES','size'),avg_sale=('SALES','mean')).dropna()
    else:
        data=df.select_dtypes(include='number').drop(columns=['ORDERNUMBER','ORDERLINENUMBER'],errors='ignore').dropna()
    if len(data)<3 or data.shape[1]<1:raise ValueError('Need at least 3 customers and numeric features')
    scaled=StandardScaler().fit_transform(data)
    ks=list(range(1,min(10,len(data)-1)+1));inertia=[]
    for k in ks:inertia.append(KMeans(n_clusters=k,random_state=42,n_init=10).fit(scaled).inertia_)
    # Largest perpendicular distance to first-last line is a simple elbow heuristic.
    if len(ks)>=3:
        coords=np.column_stack([np.array(ks)/max(ks),np.array(inertia)/max(inertia[0],1e-12)])
        a,b=coords[0],coords[-1];direction=b-a
        distances=[abs(np.linalg.det(np.stack([direction,p-a])))/max(np.linalg.norm(direction),1e-12) for p in coords]
        optimal=ks[int(np.argmax(distances[1:-1]))+1]
    else:optimal=min(2,len(data))
    optimal=max(2,optimal)
    model=KMeans(n_clusters=optimal,random_state=42,n_init=10)
    data=data.copy();data['cluster']=model.fit_predict(scaled)
    print('Suggested elbow cluster count:',optimal)
    print('Cluster sizes:\n',data.cluster.value_counts().sort_index())
    print('Cluster means:\n',data.groupby('cluster').mean().round(2))
    plt.plot(ks,inertia,'o-');plt.axvline(optimal,linestyle='--',label=f'Suggested k={optimal}')
    plt.xlabel('Number of clusters');plt.ylabel('Inertia');plt.title('Elbow method');plt.legend();plt.tight_layout()
    plt.savefig('B5_elbow.png',dpi=150);plt.close()
    data.to_csv('B5_customer_clusters.csv')
    print('Saved B5_elbow.png and B5_customer_clusters.csv')
main()
